# PCA in Practice

**Companion lesson:** https://ml-viz.vercel.app/courses/pca-dimensionality/03-pca-in-practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Compressing 8×8 digits

We PCA scikit-learn's digits (64 pixels), choose components by cumulative variance, and reconstruct.

In [ ]:
from sklearn.datasets import load_digits
X = load_digits().data            # (1797, 64)
mu = X.mean(0); Xc = X - mu
C = np.cov(Xc.T)
vals, vecs = np.linalg.eigh(C)
vals, vecs = vals[::-1], vecs[:, ::-1]   # descending

ratio = vals / vals.sum()
cum = np.cumsum(ratio)
print('components for 90% variance:', np.searchsorted(cum, 0.90) + 1)
print('components for 99% variance:', np.searchsorted(cum, 0.99) + 1)

plt.figure(figsize=(6.5, 4))
plt.plot(cum, color='#6366f1'); plt.axhline(0.95, color='#f43f5e', ls=':')
plt.xlabel('number of components'); plt.ylabel('cumulative explained variance'); plt.show()

## Reconstruction at increasing m

In [ ]:
def reconstruct(x, m):
    z = vecs[:, :m].T @ (x - mu)
    return mu + vecs[:, :m] @ z

digit = X[7]
ms = [2, 8, 21, 64]
fig, axes = plt.subplots(1, len(ms) + 1, figsize=(13, 2.8))
axes[0].imshow(digit.reshape(8, 8), cmap='magma'); axes[0].set_title('original')
for ax, m in zip(axes[1:], ms):
    ax.imshow(reconstruct(digit, m).reshape(8, 8), cmap='magma')
    err = ((reconstruct(digit, m) - digit) ** 2).mean()
    ax.set_title(f'm={m}  mse={err:.1f}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
# reconstruction error per sample averages to the sum of DISCARDED eigenvalues:
print('mean recon MSE at m=8 :', round(np.mean([((reconstruct(x, 8) - x)**2).sum() for x in X[:200]]), 1))
print('sum of discarded λ    :', round(vals[8:].sum(), 1))

## 'Eigendigits' — what the components look like

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(13, 2.2))
for i, ax in enumerate(axes):
    ax.imshow(vecs[:, i].reshape(8, 8), cmap='coolwarm'); ax.axis('off'); ax.set_title(f'PC{i+1}')
plt.tight_layout(); plt.show()
# each component is a pixel-space pattern; every digit = mean + weighted sum of these

**Try it:** reconstruction error as anomaly detector — reconstruct random-noise 'images' with m=8 and compare their error to real digits. Then whiten (`z / np.sqrt(vals[:m])`) and check the coordinates' covariance is the identity.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Project and reconstruct

The round trip that drives compression: center, project onto the top $m$ components, come back, un-center:

$$\hat{X} = X_c V_m V_m^\top + \bar{X}$$

The checks verify that keeping **all** components reconstructs exactly, more components mean lower error, and the punchline from the lesson: the per-coordinate MSE equals the **mean of the discarded eigenvalues**.

In [ ]:
def pca_reconstruct(X, V, m):
    """Reconstruct X from its projection onto the first m columns of V
    (V's columns = principal components, sorted by decreasing eigenvalue)."""
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    Xc = X - mu

    # TODO(you): the first m components
    Vm = ...

    # TODO(you): project down, map back, add the mean back
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(1)
Xr = rng.multivariate_normal([0, 0, 0], np.diag([5.0, 2.0, 0.1]), size=300)

Xc = Xr - Xr.mean(axis=0)
lams, V = np.linalg.eigh(Xc.T @ Xc / len(Xr))
lams, V = lams[::-1], V[:, ::-1]   # descending

assert np.allclose(pca_reconstruct(Xr, V, 3), Xr, atol=1e-8), "keeping all components reconstructs exactly"

err1 = np.mean((Xr - pca_reconstruct(Xr, V, 1)) ** 2)
err2 = np.mean((Xr - pca_reconstruct(Xr, V, 2)) ** 2)
assert err1 > err2, "more components -> lower error"
assert abs(err2 * 3 - lams[2]) < 0.05, "MSE x d = mean discarded eigenvalue (here just lambda_3)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def pca_reconstruct(X, V, m):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    Xc = X - mu
    Vm = V[:, :m]
    return Xc @ Vm @ Vm.T + mu
```

</details>

### Exercise 2 — Anomaly detection by reconstruction error

The "Try it" above, made concrete: normal points live near the principal subspace, so they reconstruct well; an anomaly **off** the subspace can't be represented by the top components and reconstructs badly. Compute per-point reconstruction errors and let the checks confirm a planted off-subspace point screams the loudest.

In [ ]:
def reconstruction_errors(X, V, m):
    """Per-point squared reconstruction error using the top m components."""
    X = np.asarray(X, dtype=float)

    # TODO(you): reconstruct, then sum squared error per row (axis=1)
    R = ...
    return ...

In [ ]:
# Checks — run me
X_anom = np.vstack([Xr, [[0.0, 0.0, 8.0]]])   # an outlier along the weakest direction
Xc_a = X_anom - X_anom.mean(axis=0)
_, V4 = np.linalg.eigh(Xc_a.T @ Xc_a / len(X_anom))
V4 = V4[:, ::-1]

errs = reconstruction_errors(X_anom, V4, 2)
assert int(np.argmax(errs)) == len(X_anom) - 1, "the planted off-subspace point has the largest error"
assert errs[-1] > 10 * np.median(errs), "and not by a little"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def reconstruction_errors(X, V, m):
    X = np.asarray(X, dtype=float)
    R = pca_reconstruct(X, V, m)
    return np.sum((X - R) ** 2, axis=1)
```

</details>